# Malilangwe Wildlife Detector — WAID Fine-Tuning

This notebook fine-tunes YOLOv11 on the WAID aerial wildlife dataset.

**Before running:**
1. `Runtime → Change runtime type → T4 GPU`
2. Then `Runtime → Run all`

**Time:** ~2–3 hours on T4 (100 epochs)

**Output:** `best.pt` weights will be downloaded automatically when training finishes.

## 1. Check GPU

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found. Go to Runtime → Change runtime type → T4 GPU, then re-run.')
print(result.stdout)

Thu Apr  2 15:07:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone Repo & Install Dependencies

In [2]:
import os

REPO_URL = 'https://github.com/Tadiwa-M/wildlife-detector-malilangwe.git'
REPO_DIR = '/content/wildlife-detector-malilangwe'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

Cloning into '/content/wildlife-detector-malilangwe'...
remote: Enumerating objects: 14446, done.
remote: Counting objects: 100% (14446/14446), done.
remote: Compressing objects: 100% (13968/13968), done.
remote: Total 14446 (delta 13), reused 14442 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (14446/14446), 4.12 MiB | 14.36 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Working directory: /content/wildlife-detector-malilangwe


In [3]:
!pip install -q -r requirements.txt
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 66.6 MB/s eta 0:00:00
Dependencies installed.


## 3. Download WAID Dataset (Images + Labels)

In [ ]:
WAID_DIR = '/content/wildlife-detector-malilangwe/WAID'

if not os.path.exists(os.path.join(WAID_DIR, 'WAID', 'images')):
    print('Downloading WAID dataset (~2 GB, takes a few minutes)...')
    !git clone --depth=1 https://github.com/xiaohuicui/WAID.git {WAID_DIR}/WAID_repo
    # Move images into expected structure
    !cp -r {WAID_DIR}/WAID_repo/WAID/images {WAID_DIR}/WAID/images
    print('WAID images downloaded.')
else:
    print('WAID images already present, skipping download.')

Cloning into '/content/wildlife-detector-malilangwe/WAID/WAID_repo'...
remote: Enumerating objects: 28744, done.
remote: Counting objects: 100% (28744/28744), done.
remote: Compressing objects: 100% (28288/28288), done.


## 4. Validate Dataset

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.config import load_config
from src.data.dataset import validate_dataset, get_class_distribution, generate_dataset_yaml
from src.utils.logging_setup import setup_logging
import logging

cfg = load_config()
setup_logging(cfg)

stats = validate_dataset(cfg)
print(f"\nTotal images : {stats['total_images']:,}")
print(f"Total labels : {stats['total_labels']:,}")

print('\nClass distribution (train):')
dist = get_class_distribution(cfg, split='train')
for name, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {name:<12s} {count:>8,}')

## 5. Generate Dataset YAML

In [ ]:
dataset_yaml = generate_dataset_yaml(cfg)
print('Dataset YAML written to:', dataset_yaml)

import yaml
with open(dataset_yaml) as f:
    print(yaml.safe_load(f))

## 6. Train

Training runs for 100 epochs with early stopping (patience=15).  
On a T4 GPU expect ~2–3 hours. Colab will stay alive as long as the cell is running.

In [ ]:
from ultralytics import YOLO

train_cfg = cfg.training
det_cfg = cfg.detection

model = YOLO(f"{det_cfg.model_variant}.pt")

results = model.train(
    data=str(dataset_yaml),
    epochs=int(train_cfg.epochs),
    batch=int(train_cfg.batch_size),
    imgsz=int(train_cfg.image_size),
    optimizer=str(train_cfg.optimizer),
    lr0=float(train_cfg.learning_rate),
    weight_decay=float(train_cfg.weight_decay),
    patience=int(train_cfg.patience),
    device=0,  # GPU
    # Augmentation
    hsv_h=float(train_cfg.augmentation.hsv_h),
    hsv_s=float(train_cfg.augmentation.hsv_s),
    hsv_v=float(train_cfg.augmentation.hsv_v),
    flipud=float(train_cfg.augmentation.flipud),
    fliplr=float(train_cfg.augmentation.fliplr),
    mosaic=float(train_cfg.augmentation.mosaic),
    mixup=float(train_cfg.augmentation.mixup),
    scale=float(train_cfg.augmentation.scale),
    project='runs/train',
    name='waid_yolo11n',
    exist_ok=True,
)

print('\nTraining complete!')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)', 'N/A'))

## 7. Evaluate on Test Split

In [ ]:
from pathlib import Path

best_weights = Path('runs/train/waid_yolo11n/weights/best.pt')

if best_weights.exists():
    eval_model = YOLO(str(best_weights))
    metrics = eval_model.val(
        data=str(dataset_yaml),
        split='test',
        conf=float(det_cfg.confidence_threshold),
        iou=float(det_cfg.iou_threshold),
        imgsz=int(det_cfg.image_size),
        device=0,
        plots=True,
    )
    print('\n' + '='*50)
    print(f'  mAP@50:    {metrics.box.map50:.4f}')
    print(f'  mAP@50-95: {metrics.box.map:.4f}')
    print(f'  Precision:  {metrics.box.mp:.4f}')
    print(f'  Recall:     {metrics.box.mr:.4f}')
    print('='*50)
else:
    print(f'Weights not found at {best_weights} — did training complete?')

## 8. Download Weights

This will download `best.pt` to your computer. Put it in the `weights/` folder of your local repo.

In [ ]:
from google.colab import files
import shutil

best_weights = Path('runs/train/waid_yolo11n/weights/best.pt')

if best_weights.exists():
    # Also copy to a convenient location
    shutil.copy(best_weights, '/content/waid_yolo11n_best.pt')
    files.download('/content/waid_yolo11n_best.pt')
    print('Download started!')
    print('Place the file in your local repo under: weights/waid_yolo11n_best.pt')
    print('\nThen run detection with your new model:')
    print('  python scripts/detect.py --source your_image.jpg --weights weights/waid_yolo11n_best.pt')
else:
    print(f'Weights not found at {best_weights}')

## 9. Quick Test on a Sample Image

Optional: upload one of your own aerial images and see the results.

In [ ]:
from google.colab import files as colab_files
from IPython.display import Image, display
import glob

print('Upload an aerial image to test...')
uploaded = colab_files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]
    eval_model = YOLO('runs/train/waid_yolo11n/weights/best.pt')
    results = eval_model.predict(
        source=img_path,
        conf=0.15,
        save=True,
        project='/content/test_output',
        name='result',
        exist_ok=True,
    )

    for r in results:
        print(f'Detections: {len(r.boxes)}')
        for box in r.boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f'  {r.names[cls]}: {conf:.0%}')

    # Show annotated image
    saved = glob.glob('/content/test_output/result/*.jpg') + glob.glob('/content/test_output/result/*.png')
    if saved:
        display(Image(saved[0]))